# 03 — Visualización del training set (H4)

Cómo el pipeline clásico convierte imágenes train + anotaciones GT en una matriz `(X, y)` para entrenar la SVM.

Pasos visualizados:

1. Una imagen train con su ground truth.
2. Selective Search → ~300 proposals candidatos.
3. Etiquetado por IoU contra GT: positivo (verde), negativo (rojo), ignorado (gris).
4. Histograma de IoUs — dónde caen los umbrales.
5. Estadísticas agregadas sobre 10 imágenes.

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
repo_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
sys.path.insert(0, str(repo_root / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from grocery_detection.utils.config import load_yaml
from grocery_detection.data.eda import load_coco
from grocery_detection.data.visualize import draw_bbox
from grocery_detection.classical.proposals import (
    resize_for_proposals, scale_proposals, selective_search,
)
from grocery_detection.classical.iou import iou_matrix
from grocery_detection.classical.labeling import label_proposals, BACKGROUND

data_cfg = load_yaml(repo_root / "configs" / "data.yaml")
cls_cfg = load_yaml(repo_root / "configs" / "classical.yaml")
classes_cfg = load_yaml(repo_root / "configs" / "classes.yaml")

train_coco = load_coco(repo_root / data_cfg["filtered_splits"]["train"])
img_dir = repo_root / data_cfg["paths"]["d2s_images"]
target_names = [n for g in classes_cfg["target_classes"] for n in g["items"]]
print(f"Train images: {len(train_coco['images'])}")
print(f"Annotations : {len(train_coco['annotations'])}")
print(f"Categories  : {len(target_names)} ({target_names[:3]}...)")

## 1. Una imagen train con ground truth

In [ ]:
anns_by_img = {}
for a in train_coco["annotations"]:
    anns_by_img.setdefault(a["image_id"], []).append(a)
id2img = {im["id"]: im for im in train_coco["images"]}
id2name = {c["id"]: c["name"] for c in train_coco["categories"]}

# Coger una imagen train (suelen tener 1 objeto, catálogo)
demo_img_info = train_coco["images"][0]
demo_anns = anns_by_img[demo_img_info["id"]]
print(f"Imagen: {demo_img_info['file_name']}")
print(f"GT objects: {len(demo_anns)} ({[id2name[a['category_id']] for a in demo_anns]})")

img_bgr = cv2.imread(str(img_dir / demo_img_info["file_name"]))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

gt_img = img_rgb.copy()
for a in demo_anns:
    x, y, w, h = a["bbox"]
    draw_bbox(gt_img, x, y, w, h, label=id2name[a["category_id"]], color=(0, 200, 0))

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(gt_img); ax.set_title("Ground truth"); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Selective Search produce ~300 proposals candidatos

In [ ]:
prop_cfg = cls_cfg["proposals"]
resized, scale = resize_for_proposals(img_bgr, max_side=prop_cfg["max_side"])
ss = selective_search(resized, mode=prop_cfg["mode"], max_proposals=prop_cfg["max_per_image"])
proposals = scale_proposals(ss, scale)
print(f"Generados {len(proposals)} proposals (mode={prop_cfg['mode']}, cap={prop_cfg['max_per_image']})")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img_rgb)
for (x, y, w, h) in proposals[:100]:
    ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor="#9333EA", linewidth=0.5))
ax.set_title(f"Top-100 proposals (de {len(proposals)})")
ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Etiquetado por IoU vs GT

Cada proposal recibe una etiqueta según su IoU con la GT más cercana:

- **Positivo** (verde): IoU ≥ 0.5 → entra al training como su clase GT.
- **Negativo** (rojo): IoU < 0.3 → entra al training como background (clase 0).
- **Ignorado** (gris): IoU intermedio → se descarta (zona de duda).

In [ ]:
lab_cfg = cls_cfg["labeling"]
gt_boxes = np.array([a["bbox"] for a in demo_anns], dtype=np.float32)
gt_labels = np.array([a["category_id"] for a in demo_anns], dtype=np.int64)

all_props = np.vstack([gt_boxes, proposals.astype(np.float32)])
labels, ious = label_proposals(
    all_props, gt_boxes, gt_labels,
    pos_iou=lab_cfg["pos_iou"], neg_iou=lab_cfg["neg_iou"],
)

n_pos = int((labels > 0).sum())
n_neg = int((labels == BACKGROUND).sum())
n_ign = int((labels == -1).sum())
print(f"De {len(all_props)} proposals (GT + SS):")
print(f"  positivos (verde): {n_pos}  (IoU >= {lab_cfg['pos_iou']})")
print(f"  negativos (rojo) : {n_neg}  (IoU < {lab_cfg['neg_iou']})")
print(f"  ignorados (gris) : {n_ign}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, mask_value, title, color in [
    (axes[0], labels > 0, f"Positivos ({n_pos})", (0, 200, 0)),
    (axes[1], labels == BACKGROUND, f"Negativos ({n_neg})", (220, 60, 60)),
    (axes[2], labels == -1, f"Ignorados ({n_ign})", (150, 150, 150)),
]:
    ax.imshow(img_rgb)
    sel = all_props[mask_value]
    # Limitar a 60 cajas por panel para no saturar
    for (x, y, w, h) in sel[:60]:
        rgb = (color[0]/255, color[1]/255, color[2]/255)
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=rgb, linewidth=1.2))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Distribución de IoUs

Histograma de IoU(proposal, mejor_GT) para todos los proposals. Las líneas verticales marcan los umbrales del etiquetado.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ious, bins=50, color="#9333EA", alpha=0.7, edgecolor="white")
ax.axvline(lab_cfg["neg_iou"], color="#DC2626", linestyle="--",
           label=f"neg_iou = {lab_cfg['neg_iou']}")
ax.axvline(lab_cfg["pos_iou"], color="#16A34A", linestyle="--",
           label=f"pos_iou = {lab_cfg['pos_iou']}")
ax.set_xlabel("IoU vs mejor GT")
ax.set_ylabel("# proposals")
ax.set_title("Distribución de IoUs de los proposals contra GT (esta imagen)")
ax.legend()
plt.tight_layout(); plt.show()
print("La mayoría de proposals caen muy bajo de IoU (background).")
print("Por eso submuestreamos negativos (max_neg_per_image en classical.yaml) "
      "para evitar un train set 99% background.")

## 5. Estadísticas agregadas sobre 10 imágenes

Pasamos por las primeras 10 imágenes train y contamos cuántos positivos/negativos genera cada una tras el subsampling de negativos definido en `classical.yaml`.

In [ ]:
import random
rng = random.Random(42)
ts_cfg = cls_cfg["training_set"]
max_neg = ts_cfg["max_neg_per_image"]

rows = []
for im in train_coco["images"][:10]:
    img = cv2.imread(str(img_dir / im["file_name"]))
    anns = anns_by_img.get(im["id"], [])
    gt_b = np.array([a["bbox"] for a in anns], dtype=np.float32) if anns else np.zeros((0, 4), np.float32)
    gt_l = np.array([a["category_id"] for a in anns], dtype=np.int64) if anns else np.zeros((0,), np.int64)
    resized, sc = resize_for_proposals(img, max_side=prop_cfg["max_side"])
    ss = selective_search(resized, mode=prop_cfg["mode"], max_proposals=prop_cfg["max_per_image"])
    props = scale_proposals(ss, sc).astype(np.float32)
    props = np.vstack([gt_b, props]) if gt_b.shape[0] else props
    lbls, _ = label_proposals(props, gt_b, gt_l, pos_iou=lab_cfg["pos_iou"], neg_iou=lab_cfg["neg_iou"])
    pos_n = int((lbls > 0).sum())
    neg_all = int((lbls == BACKGROUND).sum())
    neg_sampled = min(neg_all, max_neg)
    rows.append({
        "file": im["file_name"],
        "GT_objects": len(anns),
        "proposals_total": int(len(props)),
        "pos": pos_n,
        "neg_before_sampling": neg_all,
        f"neg_after_cap({max_neg})": neg_sampled,
    })
df = pd.DataFrame(rows)
df.loc["TOTAL"] = df.select_dtypes(include="number").sum()
df

In [ ]:
# Barplot pos vs neg(cap)
stats = df.drop("TOTAL")[["pos", f"neg_after_cap({max_neg})"]].copy()
stats.index = range(1, len(stats) + 1)
ax = stats.plot.bar(figsize=(10, 4), color=["#16A34A", "#DC2626"], width=0.7)
ax.set_xlabel("imagen #")
ax.set_ylabel("# muestras al training")
ax.set_title("Muestras aportadas por imagen al training set (10 primeras imágenes)")
ax.legend(["positivos", "negativos (cap)"])
plt.tight_layout(); plt.show()

## Conclusión

- Cada imagen train aporta **pocos positivos** (1-3, porque el train es catálogo) y muchos potenciales negativos (cientos).
- Submuestreamos negativos a `max_neg_per_image` para que el SVM no se ahogue en background.
- Total esperado en train completo (1650 imgs): ~2000 positivos + ~16500 negativos.
- `Hard negative mining` añadirá después los **falsos positivos más confundidores** del modelo entrenado, mejorando precision.